# CVRP Optimization: Exact Method (ILP)

## Mathematical Formulation

* **Decision Variables:** * $x_{ij} \in \{0, 1\}$: $1$ if a vehicle travels from node $i$ to $j$.
  * $u_i$: Auxiliary variable for capacity/sub-tour tracking.
* **Objective:**
  * Minimize $Z = \sum_{i} \sum_{j} c_{ij} x_{ij}$ (Total distance).
* **Constraints:**
  1. Every customer must be visited exactly once.
  2. Vehicles must return to the depot.
  3. **Capacity Constraint:** Total demand on a route $\le$ Vehicle Capacity $Q$.


## Data Parsing Component

In [2]:
!pip install scipy

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.3/30.3 MB 242.6 kB/s  0:02:05m0:00:0100:04

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip


In [4]:
!pip install pulp

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 241.9 kB/s  0:01:08m0:00:0100:03

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import distance_matrix
import pulp

def load_vrp_data(filepath):
    """Parses the .vrp file to extract nodes, demands, and capacity."""
    with open(filepath, 'r') as f:
        lines = f.readlines()

    capacity = int([l for l in lines if "CAPACITY" in l][0].split(":")[1])
    
    # Find start of sections
    coord_start = lines.index("NODE_COORD_SECTION\t\t\n") + 1
    demand_start = lines.index("DEMAND_SECTION\t\t\n") + 1
    dimension = int([l for l in lines if "DIMENSION" in l][0].split(":")[1])

    # Parse Coordinates
    coords = []
    for i in range(coord_start, coord_start + dimension):
        parts = lines[i].split()
        coords.append((float(parts[1]), float(parts[2])))

    # Parse Demands
    demands = []
    for i in range(demand_start, demand_start + dimension):
        demands.append(int(lines[i].split()[1]))

    return np.array(coords), demands, capacity

# Load your specific Kaggle files
coords, demands, capacity = load_vrp_data('data/X-n101-k25.vrp')
dist_mat = distance_matrix(coords, coords)
N = len(coords)

## ILP Model Implementation

In [ ]:
# 1. Initialize Problem
prob = pulp.LpProblem("CVRP_Exact_ILP", pulp.LpMinimize)

# 2. Decision Variables
x = pulp.LpVariable.dicts("x", [(i, j) for i in range(N) for j in range(N)], cat='Binary')
u = pulp.LpVariable.dicts("u", range(N), lowBound=0, upBound=capacity)

# 3. Objective Function: Minimize Total Distance
prob += pulp.lpSum(dist_mat[i][j] * x[i, j] for i in range(N) for j in range(N) if i != j)

# 4. Constraints
for i in range(N):
    # Every customer visited once
    prob += pulp.lpSum(x[j, i] for j in range(N) if i != j) == 1
    prob += pulp.lpSum(x[i, j] for j in range(N) if i != j) == 1

# Sub-tour elimination & Capacity (MTZ Formulation)
for i in range(1, N):
    for j in range(1, N):
        if i != j:
            prob += u[i] - u[j] + capacity * x[i, j] <= capacity - demands[j]

for i in range(1, N):
    prob += u[i] >= demands[i]

# 5. Solver with 15-minute time limit (as the problem is NP-Hard)
solver = pulp.PULP_CBC_CMD(timeLimit=900, msg=1)
prob.solve(solver)